# Data Fusion 2025 — Задача 2 "4cast"
## Прогнозирование еженедельных переводов юридических лиц

**Задача:** предсказать суммарные переводы 51 963 клиентов банка за 12 недель (118–129),  
используя историю за 118 недель (0–117).

**Метрика:** средний по клиентам RMSLE:
$$\overline{RMSLE} = \frac{1}{N}\sum_{i=1}^N \sqrt{\frac{1}{T}\sum_{t=1}^T (\log(1+y_{it}) - \log(1+\hat{y}_{it}))^2}$$

**Public leaderboard:** недели 118–121 (4 из 12 прогнозных).

---


## Скачивание данных

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import lightgbm as lgb
import warnings
import json
from pathlib import Path
import gc, time

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (14, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.4
sns.set_palette("tab10")

DATA_DIR = Path("data")
FIG_DIR  = Path("figures1")
FIG_DIR.mkdir(exist_ok=True)

# Splits (из calendar_extended.csv)
TRAIN_WEEKS    = list(range(0, 106))     # 0–105  → train
VAL_PUB_WEEKS  = list(range(106, 110))  # 106–109 → validation_public
VAL_PRIV_WEEKS = list(range(110, 118))  # 110–117 → validation_private
PUBLIC_WEEKS   = list(range(118, 122))  # 118–121 → public leaderboard
PRIVATE_WEEKS  = list(range(122, 130))  # 122–129 → private leaderboard
FORECAST_WEEKS = list(range(118, 130))  # все 12 недель прогноза

print("Config loaded.")


Config loaded.


In [4]:
print("Loading data...")
ts_train = pd.read_parquet(DATA_DIR / "target_series_extended.parquet")       # weeks 0–117
calendar = pd.read_csv(DATA_DIR / "calendar_extended.csv", parse_dates=["date"])
sample   = pd.read_csv(DATA_DIR / "sample_submit_extended.csv")
profiles = pd.read_parquet(DATA_DIR / "profiles_extended.parquet")

ts_train["week"] = ts_train["week"].astype(int)

TX_FILES = sorted(DATA_DIR.glob("transactions_[1-5].parquet"))
print(f"Found {len(TX_FILES)} transaction files: {[f.name for f in TX_FILES]}")

t0 = time.time()
trns_chunks = []
for f in TX_FILES:
    print(f"Reading {f.name} ...")
    chunk = pd.read_parquet(f)
    print(f"  -> {len(chunk):,} rows")
    trns_chunks.append(chunk)

trns_raw = pd.concat(trns_chunks, ignore_index=True)
del trns_chunks
gc.collect()

Loading data...
Found 5 transaction files: ['transactions_1.parquet', 'transactions_2.parquet', 'transactions_3.parquet', 'transactions_4.parquet', 'transactions_5.parquet']
Reading transactions_1.parquet ...
  -> 52,089,892 rows
Reading transactions_2.parquet ...
  -> 55,045,806 rows
Reading transactions_3.parquet ...
  -> 59,510,350 rows
Reading transactions_4.parquet ...
  -> 61,492,247 rows
Reading transactions_5.parquet ...
  -> 30,036,951 rows


0

## Подготовка данных 

Разбиваем датасет с транзакциями на 117 файлов, по неделе на файл для оптимизации. 

In [5]:
trns = trns_raw.copy()

In [4]:
#trns = trns.iloc[:50000]

In [6]:
trns["date"] = pd.to_datetime(trns["date"], utc=True).dt.tz_localize(None).dt.normalize()

all_inn = ts_train["inn_id"].unique()

cal_117 = calendar.loc[calendar["week"] <= 117].copy()
cal_117["date"] = pd.to_datetime(cal_117["date"]).dt.tz_localize(None).dt.normalize()
all_dates = np.sort(cal_117["date"].drop_duplicates().values)

mi = pd.MultiIndex.from_product([all_inn, all_dates], names=["doc_payer_inn", "date"])
new_trns = mi.to_frame(index=False)

new_trns["doc_payee_inn"] = ""
new_trns["trns_count"] = 1.0
new_trns["trns_amount"] = 0
new_trns["doc_payer_bank_name_encoded"] = 51
new_trns["doc_payee_bank_name_encoded"] = 0
new_trns["doc_payer_bank_name_flag"] = 1
new_trns["doc_payee_bank_name_flag"] = 0
new_trns["trns_class_encoded"] = 0

trns = pd.concat([trns, new_trns], ignore_index=True)

trns = pd.merge(trns, calendar, on='date', how='left')

allowed = ts_train["inn_id"].unique()
trns = trns[trns["doc_payer_inn"].isin(allowed)]

In [6]:
# построение ряда с суммами переводов по неделям только со счетов втб 
trns_vtb = trns[trns["doc_payer_bank_name_flag"] == 1]
trns_vtb = trns_vtb.groupby(["week", "doc_payer_inn"], as_index=False, sort=False).agg({
    "trns_amount": "sum",
    "part": "first",
}).rename(columns={
                 "doc_payer_inn": "inn_id",
                 "trns_amount": "target",
             })

trns_vtb.drop(columns=["part"], inplace=True)

In [7]:
trns = trns.groupby(["week", "doc_payer_inn"], as_index=False, sort=False).agg({
    "trns_amount": "sum",
    "part": "first",
}).rename(columns={
                 "doc_payer_inn": "inn_id",
                 "trns_amount": "target",
             })

In [9]:
# trns = pd.merge(trns, trns_vtb, on=["week", "inn_id"], how="left")  


In [8]:
trns.head()

,week,inn_id,target,part
0,0,inn1000051,4.221593e+05,train
1,0,inn1000367,3.994532e+05,train
2,0,inn1000484,9.672847e+05,train
3,0,inn1000624,1.684209e+05,train
4,0,inn1000884,5.724560e+06,train


In [9]:
del trns_vtb
del new_trns
del cal_117
gc.collect()

NameError: name 'trns_vtb' is not defined

In [ ]:
expected_weeks = sorted(trns["week"].astype(int).unique().tolist())

expected_len = trns["inn_id"].nunique()

dup_mask = trns.duplicated(subset=["inn_id", "week"], keep=False)
if dup_mask.any():
    dup_count = int(dup_mask.sum())
    raise ValueError(f"Найдены дубли по (inn_id, week): {dup_count} строк")

min_week, max_week = expected_weeks[0], expected_weeks[-1]
full_range = list(range(min_week, max_week + 1))
if expected_weeks != full_range:
    missing_weeks = sorted(set(full_range) - set(expected_weeks))
    raise ValueError(f"Неполный диапазон недель. Отсутствуют: {missing_weeks}")

out_dir = Path("transactions_chunks")
out_dir.mkdir(parents=True, exist_ok=True)

trns_sorted = trns.sort_values(["week", "inn_id"]).reset_index(drop=True)
errors = []
written_files = 0

for week, chunk in trns_sorted.groupby("week", sort=True):
    chunk_len = len(chunk)
    if chunk_len != expected_len:
        errors.append(f"week={int(week)}: len={chunk_len}, expected={expected_len}")
        continue

    out_path = out_dir / f"week_{int(week):03d}.parquet"
    chunk.to_parquet(out_path, index=False)
    written_files += 1

if errors:
    preview = "\n".join(errors[:10])
    raise ValueError(f"Обнаружены недели с неверной длиной чанка:\n{preview}")

print(f"Done: записано {written_files} файлов в {out_dir.resolve()}")

Done: записано 118 файлов в /Users/aeseverenkova/Desktop/bank_forecasting/transactions_chunks


## Построение признаков

Хотим в признаках зафиксировать связь предыдущих событий на будующие. Для этого будем использовать лаги.

In [12]:
def _lags_row_at_cutoff(n: int, weeks: np.ndarray, y: np.ndarray, max_lag: int = 70):
    """Одна строка лагов для недели n (эквивалент shift)"""

    mask = (weeks <= n) & (weeks >= n - max_lag)
    idx = np.flatnonzero(mask)

    w = weeks[idx] # отфильтрованные недели в нужном диапозоне 
    yy = y[idx] # отфильтрованные таргеты в нужном диапозоне 
    pos = int(np.where(w == n)[0][0])

    row = {}
    for lag in range(1, max_lag + 1):
        j = pos - lag
        row[f"lag_{lag}"] = float(yy[j])
        
    return row

def get_features(n, df) -> pd.DataFrame:
    """
    n - номер недели
    df - датафрейм с транзакциями одного inn_id
    """

    max_lag = 70

    g = df.sort_values("week")
    weeks = g["week"].to_numpy()
    y = g["target"].to_numpy(dtype=float)
    row = _lags_row_at_cutoff(n, weeks, y, max_lag)
    
    return pd.DataFrame([row]) if row else pd.DataFrame()


In [13]:
def build_features_horizons_for_inn(
    n_target: int, k: int, inn_df: pd.DataFrame, max_lag: int = 70
) -> pd.DataFrame:
    """Все горизонты для одного ИНН """

    g = inn_df.sort_values("week")
    weeks = g["week"].to_numpy()
    y = g["target"].to_numpy(dtype=float)

    rows = []
    for i in range(k):
        cutoff = n_target - i
        row = _lags_row_at_cutoff(cutoff, weeks, y, max_lag)
        row['inn_id'] = inn_df['inn_id'].iloc[0] # чтобы потом по inn доклеить profiles 
        rows.append(row) 
        
    return pd.DataFrame(rows) if rows else pd.DataFrame()

Представим, что мы хотим предсказать на одну неделю вперед. У нас есть данные 0-117 недель. Тогда мы будем считать 117 неделю таргетом, а строить признаки на 0-116 неделях. 

Если мы хотим предсказать 2 неделю, таргетом снова считаем 117 неделю, но признаки уже строим на 0-115 неделях. 

Если мы хотим предсказать 3 неделю, мы строрим признаки на 0-114 неделях. 

Тогда чтобы предсказать неделю k надо строить признаки на неделях 117-k. Так мы будем показывать как влияют предыдущие значения на будующую неделю номер k. 

Предсказание на 12 недель: предсказание на 1, предсказание на 2, .... предсказание на 12 неделю.

In [14]:
train = pd.DataFrame()
n = 117 # текущая неделя 
k = 12 # горизонт предсказания

train_list = []

df_week_n = trns.loc[trns["week"] == n, ["inn_id", "target"]]
inn_subset = df_week_n["inn_id"].unique()
inn_groups = {gid: g for gid, g in trns.groupby("inn_id", sort=False)}

train_list = []
for inn in inn_subset:
    inn_df = inn_groups.get(inn)

    t_series = df_week_n.loc[df_week_n["inn_id"] == inn, "target"]
    t_val = t_series.iloc[0]

    f_block = build_features_horizons_for_inn(n, k, inn_df)
    f_block["target"] = t_val
    train_list.append(f_block)
        
train = pd.concat(train_list, ignore_index=True) if train_list else pd.DataFrame() 

# добавление категориальных признаков 
train = pd.merge(train, profiles, on="inn_id", how="left")
train.drop(columns=['inn_id'], inplace=True)

# логарифмирование признаков
numeric_cols = train.select_dtypes(include=[np.number]).columns
train[numeric_cols] = train[numeric_cols].applymap(lambda x: np.log(x + 1))

train.head()

,lag_1,lag_2,lag_3,lag_4,lag_5,lag_6,lag_7,lag_8,lag_9,lag_10,...,lag_67,lag_68,lag_69,lag_70,target,report_date,ipul,id_region,main_okved_group,diff_datopen_report_date_flg
0,11.82079,12.645852,13.426701,12.470371,10.357664,10.593899,15.600336,14.73538,9.982253,10.961955,...,11.399904,12.089898,11.090265,13.04886,11.195589,2023-10-01,ip,45,68,2_4y
1,11.82079,12.645852,13.426701,12.470371,10.357664,10.593899,15.600336,14.73538,9.982253,10.961955,...,11.399904,12.089898,11.090265,13.04886,11.195589,2023-11-01,ip,45,68,2_4y
2,11.82079,12.645852,13.426701,12.470371,10.357664,10.593899,15.600336,14.73538,9.982253,10.961955,...,11.399904,12.089898,11.090265,13.04886,11.195589,2023-12-01,ip,45,68,2_4y
3,11.82079,12.645852,13.426701,12.470371,10.357664,10.593899,15.600336,14.73538,9.982253,10.961955,...,11.399904,12.089898,11.090265,13.04886,11.195589,2024-01-01,ip,45,68,2_4y
4,11.82079,12.645852,13.426701,12.470371,10.357664,10.593899,15.600336,14.73538,9.982253,10.961955,...,11.399904,12.089898,11.090265,13.04886,11.195589,2024-02-01,ip,45,68,2_4y


In [15]:
del train
gc.collect()

0

Чтобы увеличить обучающую выборку, попробуем не фиксировать таргетную неделю на n = 117. Построим по схеме выше признаки для 5 таргенных недель (117...97). 

In [16]:
def sample_n(n, df) -> pd.DataFrame:
    """ Собрать обучающую выборку для недели n """
    k = 12 # горизонт предсказания

    df_week_n = df.loc[df["week"] == n, ["inn_id", "target"]]
    inn_subset = df_week_n["inn_id"].unique()
    inn_groups = {gid: g for gid, g in df.groupby("inn_id", sort=False)}

    train_list = []
    for inn in inn_subset:
        inn_df = inn_groups.get(inn)

        t_series = df_week_n.loc[df_week_n["inn_id"] == inn, "target"]
        t_val = t_series.iloc[0]

        f_block = build_features_horizons_for_inn(n, k, inn_df)
        f_block["target"] = t_val
        train_list.append(f_block)

    return pd.concat(train_list, ignore_index=True) if train_list else pd.DataFrame()

In [17]:
N = 5
train_parts = []

for i in range(N):
    n = 117 - i
    train_parts.append(sample_n(n, trns))

train = pd.concat(train_parts, ignore_index=True) if train_parts else pd.DataFrame()

# добавление категориальных признаков 
train = pd.merge(train, profiles, on="inn_id", how="left")
train.drop(columns=['inn_id'], inplace=True)

# логарифмирование признаков
numeric_cols = train.select_dtypes(include=[np.number]).columns
train[numeric_cols] = train[numeric_cols].applymap(lambda x: np.log(x + 1))

train.head()

,lag_1,lag_2,lag_3,lag_4,lag_5,lag_6,lag_7,lag_8,lag_9,lag_10,...,lag_67,lag_68,lag_69,lag_70,target,report_date,ipul,id_region,main_okved_group,diff_datopen_report_date_flg
0,11.82079,12.645852,13.426701,12.470371,10.357664,10.593899,15.600336,14.73538,9.982253,10.961955,...,11.399904,12.089898,11.090265,13.04886,11.195589,2023-10-01,ip,45,68,2_4y
1,11.82079,12.645852,13.426701,12.470371,10.357664,10.593899,15.600336,14.73538,9.982253,10.961955,...,11.399904,12.089898,11.090265,13.04886,11.195589,2023-11-01,ip,45,68,2_4y
2,11.82079,12.645852,13.426701,12.470371,10.357664,10.593899,15.600336,14.73538,9.982253,10.961955,...,11.399904,12.089898,11.090265,13.04886,11.195589,2023-12-01,ip,45,68,2_4y
3,11.82079,12.645852,13.426701,12.470371,10.357664,10.593899,15.600336,14.73538,9.982253,10.961955,...,11.399904,12.089898,11.090265,13.04886,11.195589,2024-01-01,ip,45,68,2_4y
4,11.82079,12.645852,13.426701,12.470371,10.357664,10.593899,15.600336,14.73538,9.982253,10.961955,...,11.399904,12.089898,11.090265,13.04886,11.195589,2024-02-01,ip,45,68,2_4y


In [20]:
train.to_parquet('train_extended.parquet')

In [ ]:
# train.to_parquet('train.parquet')

## Категориальные признаки

In [10]:
import category_encoders as ce

In [24]:
pr = profiles.copy()

pr["date"] = pd.to_datetime(pr["report_date"], utc=True).dt.tz_localize(None).dt.normalize()
pr = pd.merge(pr, calendar, on='date', how='left')
pr = pr[['inn_id', 'ipul', 'id_region', 'main_okved_group', 'diff_datopen_report_date_flg', 'week']]

pr.groupby(["week", "inn_id"], as_index=False, sort=False).agg({
    "ipul": "first",
    "id_region":"first",
    "main_okved_group":"first",
    "diff_datopen_report_date_flg":"first"
})

pr_y = trns[['target', 'week', 'inn_id']].copy()
pr_y = pr_y.groupby(["week", "inn_id"], as_index=False, sort=False)['target'].sum()
pr = pd.merge(pr_y, pr, on=['inn_id', 'week'], how='left')

pr.head()

,week,inn_id,target,ipul,id_region,main_okved_group,diff_datopen_report_date_flg
0,0,inn1000051,4.221593e+05,NaN,NaN,NaN,NaN
1,0,inn1000367,3.994532e+05,NaN,NaN,NaN,NaN
2,0,inn1000484,9.672847e+05,NaN,NaN,NaN,NaN
3,0,inn1000624,1.684209e+05,NaN,NaN,NaN,NaN
4,0,inn1000884,5.724560e+06,NaN,NaN,NaN,NaN


In [26]:
len(pr)

6131634

In [27]:
pr.isna().sum()

week                                  0
inn_id                                0
target                                0
ipul                            5456234
id_region                       5457801
main_okved_group                5461153
diff_datopen_report_date_flg    5456234
dtype: int64

In [28]:
pr.notna().sum()

week                            6131634
inn_id                          6131634
target                          6131634
ipul                             675400
id_region                        673833
main_okved_group                 670481
diff_datopen_report_date_flg     675400
dtype: int64

In [29]:
a = pr.dropna()
len(a)

670481

In [30]:
len(trns)

6131634

In [1]:
pr.dropna(0)

cat_cols = ['diff_datopen_report_date_flg', 'main_okved_group', 'id_region', 'ipul']
encoder = ce.TargetEncoder(cols=cat_cols)
pr[cat_cols] = encoder.fit_transform(pr[cat_cols], pr['target'])
pr.head()



NameError: name 'pr' is not defined

## Обучение

In [21]:
!pip3 install xgboost

  Using cached xgboost-3.2.0-py3-none-macosx_12_0_arm64.whl (2.3 MB)

[notice] A new release of pip available: 22.3.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [22]:
import xgboost as xgb
from sklearn.metrics import mean_squared_log_error

In [23]:
train = train.dropna()

y = train['target']
X = train.drop(columns=['target'])

split = int(len(train) * 0.8)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

model = xgb.XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
)

model.fit(X_train, y_train)
pred_log = model.predict(X_test)

pred[numeric_cols] = np.expm1(pred_log[numeric_cols])-1

ValueError: DataFrame.dtypes for data must be int, float, bool or category. When categorical type is supplied, the experimental DMatrix parameter`enable_categorical` must be set to `True`.  Invalid columns:report_date: object, ipul: object, id_region: object, main_okved_group: object, diff_datopen_report_date_flg: object

In [29]:
# pred[pred < 0] = 0.0

In [30]:
rmsle = np.sqrt(mean_squared_log_error(y_test, pred))
print(f"RMSLE: {rmsle}")

RMSLE: 2.345562261514614


In [31]:
pred

array([2074201.4 , 2197377.5 , 2419136.  , ...,  934271.8 ,  883952.7 ,
        983292.06], shape=(623556,), dtype=float32)